# 01 — Bronze Orders Load

## Overview
This lab implements the **first layer of the medallion architecture**: a parameterised, append-only ingestion pipeline that lands raw CSV order data into a Bronze Delta table on OneLake.

**What you will learn**
- How to accept Data Factory pipeline parameters inside a Fabric notebook
- How to enforce an explicit schema at read time to catch type mismatches early
- How to attach audit metadata (ingest date, timestamp, source file) for incremental watermarking downstream
- How to configure Delta Lake auto-compaction to avoid small-files accumulation
- How to validate partition writes with a count assertion

**Prerequisites:** `lh_advanced_scenarios` Lakehouse created; `Files/landing/orders/<date>/` folder populated with a sample CSV.  
**Triggered by:** `pl_bronze_orders_ingest` Data Factory pipeline (parameters: `p_landing_path`, `p_ingest_date`).

### Cell 1 — Pipeline parameters
These two variables are **injected at runtime by the Data Factory pipeline**. `p_landing_path` points to the OneLake `Files/` folder where the source system drops daily CSV files. `p_ingest_date` is the watermark date (format `YYYY-MM-DD`) used to select the correct sub-folder and tag every row so downstream Silver jobs can filter by date without scanning the whole table.

In [ ]:
# ── Pipeline parameters (injected by Data Factory) ──────────────────────────
p_landing_path = "Files/landing/orders"
p_ingest_date  = "1900-01-01"   # overridden at runtime

### Cell 2 — Spark & Delta configuration
Tunes Spark for small-to-medium batch sizes. `spark.sql.shuffle.partitions=8` reduces the overhead of the default 200 partitions on small datasets. `optimizeWrite` lets Delta automatically coalesce small write tasks into right-sized Parquet files, and `autoCompact` triggers lightweight compaction after each write — both prevent the small-files problem without requiring a manual `OPTIMIZE` job.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType, DecimalType

LAKEHOUSE     = "lh_advanced_scenarios"
TARGET_TABLE  = f"{LAKEHOUSE}.bronze.orders_raw"
SOURCE_PATH   = f"{p_landing_path}/{p_ingest_date}/"  # partition folder

### Cell 3 — Explicit schema definition
Declares a **fixed StructType schema** rather than relying on CSV type inference. Enforcing types at read time catches malformed source data early — for example, a non-numeric `amount` or an unparseable `updated_at` timestamp will fail immediately here rather than silently coercing to `null` and corrupting the Bronze table. Setting `inferSchema=false` in the reader (next cell) ensures this schema is used exclusively.

In [ ]:
# ── Schema definition ────────────────────────────────────────────────────────
order_schema = StructType([
    StructField("order_id",     StringType(),       False),
    StructField("customer_id",  StringType(),       False),
    StructField("order_date",   DateType(),         True),
    StructField("updated_at",   TimestampType(),    False),
    StructField("status",       StringType(),       True),
    StructField("amount",       DecimalType(10, 2), True),
])

### Cell 4 — Read landing CSV
Reads the partitioned source folder using the enforced schema. `header=true` skips the CSV header row. The row count and schema printout act as an **operational sanity check** — a zero-row count at this point means the source file is missing or the path is wrong, and it is better to find out now than after the write.

In [ ]:
# ── Read landing CSV ─────────────────────────────────────────────────────────
df_raw = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "false")
         .schema(order_schema)
         .load(SOURCE_PATH)
)

print(f"Rows read: {df_raw.count():,}")
df_raw.printSchema()

### Cell 5 — Audit columns
Attaches three audit columns to every row without modifying source data:
- `_ingest_date` — the logical business date, used as the **Delta partition key** so downstream jobs scan only one day's partition.
- `_ingest_ts` — wall-clock ingestion timestamp, used as the **incremental watermark** by Silver notebooks to find unprocessed rows.
- `_source_file` — OneLake path of the original file, enabling precise row-level lineage and targeted re-loads when a source file is corrected.

In [ ]:
# ── Add audit columns ────────────────────────────────────────────────────────
df_staged = (
    df_raw
    .withColumn("_ingest_date",   F.lit(p_ingest_date).cast(DateType()))
    .withColumn("_ingest_ts",     F.current_timestamp())
    .withColumn("_source_file",   F.input_file_name())
)

### Cell 6 — Append to Bronze Delta table
Writes to the Delta table in **append mode** — Bronze is an immutable raw log; rows are never updated or deleted here. `partitionBy("_ingest_date")` creates a Hive-style folder structure (`_ingest_date=2025-01-15/`) that enables **partition pruning** in downstream Spark and SQL queries. `mergeSchema=true` safely accommodates new columns added by the source system without breaking the pipeline.

In [ ]:
# ── Append to Bronze Delta table ─────────────────────────────────────────────
(
    df_staged.write
             .format("delta")
             .mode("append")
             .option("mergeSchema", "true")
             .partitionBy("_ingest_date")
             .saveAsTable(TARGET_TABLE)
)

print(f"Write complete → {TARGET_TABLE}")

### Cell 7 — Partition validation
Queries the specific `_ingest_date` partition just written and asserts at least one row exists. A zero-row result means either: (a) the source CSV was empty, (b) the folder path is wrong, or (c) the Data Factory `p_ingest_date` parameter was malformed. **Failing loudly here prevents a silent empty partition** from propagating to Silver where it would look like a successful run with no data.

In [ ]:
# ── Validation: row count for today's partition ───────────────────────────────
count = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {TARGET_TABLE} WHERE _ingest_date = '{p_ingest_date}'"
).collect()[0]["cnt"]

print(f"Partition {p_ingest_date} row count: {count:,}")
assert count > 0, "ERROR: No rows written — check source path and watermark."